In [3]:
import cv2
import mediapipe as mp
import time
from datetime import datetime
import numpy as np
from ultralytics import YOLO  

# Mediapipe setup
mp_face_mesh = mp.solutions.face_mesh
mp_pose = mp.solutions.pose

face_mesh = mp_face_mesh.FaceMesh(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)
pose = mp_pose.Pose(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

# YOLO weapon model

weapon_model = YOLO("yolov8_updated.pt") 

# newly added

def landmark_visible(lm, edges, w, h):
    x = int(lm.x * w)
    y = int(lm.y * h)

    r = 6
    patch = edges[max(0, y-r):min(h, y+r),
                  max(0, x-r):min(w, x+r)]

    if patch.size == 0:
        return False

    return np.mean(patch) > 3

# changed function 
def calculate_face_coverage(face_landmarks, edges, w, h):
    critical_landmarks = [33, 133, 362, 263, 1, 2, 98, 324, 13, 14]
    visible_landmarks = 0

    for idx in critical_landmarks:
        if landmark_visible(face_landmarks[idx], edges, w, h):
            visible_landmarks += 1

    coverage = (visible_landmarks / len(critical_landmarks)) * 100
    return coverage

# Pose behavior logic 
def analyze_behavior(pose_landmarks):
    left_hand_y = pose_landmarks[mp_pose.PoseLandmark.LEFT_WRIST].y
    right_hand_y = pose_landmarks[mp_pose.PoseLandmark.RIGHT_WRIST].y
    nose_y = pose_landmarks[mp_pose.PoseLandmark.NOSE].y

    if left_hand_y < nose_y or right_hand_y < nose_y:
        return True  
    return False

# Webcam
cap = cv2.VideoCapture(r"C:\Users\Andhavarapu Jahnavi\Desktop\IntelliGuard Multi-Modal AI Threat Detection System\sample.mp4")

alert_message = "Warning: Potential Threat!"
weapon_alert_message = "Weapon Detected!"
alert_duration = 3
last_alert_time = 0

# Main loop

while True:
    success, frame = cap.read()
    if not success:
        break

    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray_frame, 80, 160)  # New line added

    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    face_results = face_mesh.process(image_rgb)
    pose_results = pose.process(image_rgb)

    h, w = frame.shape[:2] # New line added

    unusual_behavior_detected = False

    # Face coverage (NOW CHANGES)

    if face_results.multi_face_landmarks:
        for face_landmarks in face_results.multi_face_landmarks:
            coverage = calculate_face_coverage(
                face_landmarks.landmark, edges, w, h
            )

            print(f"Coverage: {coverage:.2f}%")

            if coverage < 20:
                current_time = time.time()
                if current_time - last_alert_time > alert_duration:
                    cv2.putText(frame, alert_message, (50, 50),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

                    filename = f"warning_image_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jpg"
                    cv2.imwrite(filename, frame)
                    last_alert_time = current_time

    # ==============================
    # Unusual behavior (UNCHANGED)
    # ==============================
    if pose_results.pose_landmarks:
        unusual_behavior_detected = analyze_behavior(
            pose_results.pose_landmarks.landmark
        )

    if unusual_behavior_detected:
        cv2.putText(frame, "Unusual Behavior Detected!", (50, 100),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        filename = f"behavior_warning_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jpg"
        cv2.imwrite(filename, frame)

    # ==============================
    # Weapon detection (UNCHANGED)
    # ==============================
    weapon_results = weapon_model(frame, stream=True)

    for result in weapon_results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            confidence = box.conf[0]
            label = result.names[int(box.cls[0])]

            print(label, float(confidence))

            if label in ['knife', 'pistol'] and confidence > 0.25:
                cv2.rectangle(frame, (x1, y1), (x2, y2),
                              (0, 0, 255), 2)

                cv2.putText(frame, f"{label} {confidence:.2f}",
                            (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                            (0, 0, 255), 2)

                cv2.putText(frame, weapon_alert_message, (50, 150),
                            cv2.FONT_HERSHEY_SIMPLEX, 1,
                            (0, 0, 255), 2)

                filename = f"weapon_detected_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jpg"
                cv2.imwrite(filename, frame)

    cv2.imshow('Surveillance System', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()



0: 384x640 (no detections), 101.4ms
Speed: 4.7ms preprocess, 101.4ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 99.9ms
Speed: 3.3ms preprocess, 99.9ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 109.3ms
Speed: 3.7ms preprocess, 109.3ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 95.8ms
Speed: 4.6ms preprocess, 95.8ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 110.1ms
Speed: 2.9ms preprocess, 110.1ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 123.9ms
Speed: 3.6ms preprocess, 123.9ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 130.6ms
Speed: 5.6ms preprocess, 130.6ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 92.4ms
Speed: 3.2ms preprocess

In [4]:
import cv2
import mediapipe as mp
import time
from datetime import datetime
import numpy as np
from ultralytics import YOLO  

# Mediapipe setup
mp_face_mesh = mp.solutions.face_mesh
mp_pose = mp.solutions.pose

face_mesh = mp_face_mesh.FaceMesh(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)
pose = mp_pose.Pose(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

# YOLO weapon model

weapon_model = YOLO("yolov8_updated.pt") 

# newly added

def landmark_visible(lm, edges, w, h):
    x = int(lm.x * w)
    y = int(lm.y * h)

    r = 6
    patch = edges[max(0, y-r):min(h, y+r),
                  max(0, x-r):min(w, x+r)]

    if patch.size == 0:
        return False

    return np.mean(patch) > 3

# changed function 
def calculate_face_coverage(face_landmarks, edges, w, h):
    critical_landmarks = [33, 133, 362, 263, 1, 2, 98, 324, 13, 14]
    visible_landmarks = 0

    for idx in critical_landmarks:
        if landmark_visible(face_landmarks[idx], edges, w, h):
            visible_landmarks += 1

    coverage = (visible_landmarks / len(critical_landmarks)) * 100
    return coverage

# Pose behavior logic 
def analyze_behavior(pose_landmarks):
    left_hand_y = pose_landmarks[mp_pose.PoseLandmark.LEFT_WRIST].y
    right_hand_y = pose_landmarks[mp_pose.PoseLandmark.RIGHT_WRIST].y
    nose_y = pose_landmarks[mp_pose.PoseLandmark.NOSE].y

    if left_hand_y < nose_y or right_hand_y < nose_y:
        return True  
    return False

# Webcam
cap = cv2.VideoCapture(0)

alert_message = "Warning: Potential Threat!"
weapon_alert_message = "Weapon Detected!"
alert_duration = 3
last_alert_time = 0

# Main loop

while True:
    success, frame = cap.read()
    if not success:
        break

    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray_frame, 80, 160)  # New line added

    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    face_results = face_mesh.process(image_rgb)
    pose_results = pose.process(image_rgb)

    h, w = frame.shape[:2] # New line added

    unusual_behavior_detected = False

    # Face coverage (NOW CHANGES)

    if face_results.multi_face_landmarks:
        for face_landmarks in face_results.multi_face_landmarks:
            coverage = calculate_face_coverage(
                face_landmarks.landmark, edges, w, h
            )

            print(f"Coverage: {coverage:.2f}%")

            if coverage < 20:
                current_time = time.time()
                if current_time - last_alert_time > alert_duration:
                    cv2.putText(frame, alert_message, (50, 50),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

                    filename = f"warning_image_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jpg"
                    cv2.imwrite(filename, frame)
                    last_alert_time = current_time

    # ==============================
    # Unusual behavior (UNCHANGED)
    # ==============================
    if pose_results.pose_landmarks:
        unusual_behavior_detected = analyze_behavior(
            pose_results.pose_landmarks.landmark
        )

    if unusual_behavior_detected:
        cv2.putText(frame, "Unusual Behavior Detected!", (50, 100),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        filename = f"behavior_warning_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jpg"
        cv2.imwrite(filename, frame)

    # ==============================
    # Weapon detection (UNCHANGED)
    # ==============================
    weapon_results = weapon_model(frame, stream=True)

    for result in weapon_results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            confidence = box.conf[0]
            label = result.names[int(box.cls[0])]

            print(label, float(confidence))

            if label in ['knife', 'pistol'] and confidence > 0.25:
                cv2.rectangle(frame, (x1, y1), (x2, y2),
                              (0, 0, 255), 2)

                cv2.putText(frame, f"{label} {confidence:.2f}",
                            (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                            (0, 0, 255), 2)

                cv2.putText(frame, weapon_alert_message, (50, 150),
                            cv2.FONT_HERSHEY_SIMPLEX, 1,
                            (0, 0, 255), 2)

                filename = f"weapon_detected_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jpg"
                cv2.imwrite(filename, frame)

    cv2.imshow('Surveillance System', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()



0: 480x640 (no detections), 190.9ms
Speed: 16.2ms preprocess, 190.9ms inference, 7.3ms postprocess per image at shape (1, 3, 480, 640)
Coverage: 50.00%

0: 480x640 (no detections), 104.0ms
Speed: 2.9ms preprocess, 104.0ms inference, 2.3ms postprocess per image at shape (1, 3, 480, 640)
Coverage: 0.00%

0: 480x640 (no detections), 102.8ms
Speed: 2.0ms preprocess, 102.8ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)
Coverage: 0.00%

0: 480x640 (no detections), 101.1ms
Speed: 1.3ms preprocess, 101.1ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)
Coverage: 0.00%

0: 480x640 (no detections), 92.7ms
Speed: 1.6ms preprocess, 92.7ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)
Coverage: 0.00%

0: 480x640 1 pistol, 105.7ms
pistol 0.44860678911209106
Speed: 1.8ms preprocess, 105.7ms inference, 21.0ms postprocess per image at shape (1, 3, 480, 640)
Coverage: 0.00%

0: 480x640 (no detections), 119.0ms
Speed: 2.5ms preprocess, 119.0ms inf